In [1]:
import numpy as np

### RNN 계층 구현  

In [ ]:
class RNN:
  def __init__(self, Wx, Wh, b):  # Wx: 입력 x에 대한 가중치, Wh: 이전 상태 h에 대한 가중치, b: 편향
    self.params = [Wx, Wh, b]   # 입력받은 Wx, Wh, b를 params에 리스트로 저장
    self.grads = [np.zeros_like(Wx), np.zeros_like(Wh), np.zeros_like(b)]  # 기울기 초기화
    self.cache = None           # 순전파 시 입력된 x, 이전 상태 h, 다음 상태 h를 저장할 cache
  
  # 순전파
  def forward(self, x, h_prev):                 # x: 입력, h_prev(h_t-1): 이전 상태
    Wx, Wh, b = self.params                     # Wx, Wh, b를 params에서 가져옴
    t = np.dot(h_prev, Wh) + np.dot(x, Wx) + b  # h_prev, x, b를 행렬 곱하고 편향을 더해 t를 구함
    h_next = np.tanh(t)                         # tanh 함수를 통과시켜 h_next를 구함, h_t = tanh(h_t-1 * Wh + x_t * Wx + b)
    self.cache = (x, h_prev, h_next)            # x, h_prev, h_next를 cache에 저장
    return h_next                               # h_next(h_t) 반환
  
  # 역전파
  def backward(self, dh_next):        # dh_next: 출력 쪽에서 전해지는 기울기
    Wx, Wh, b = self.params           # Wx, Wh, b를 params에서 가져옴
    x, h_prev, h_next = self.cache    # cache에서 x, h_prev, h_next를 가져옴
    
    dt = dh_next * (1 - h_next ** 2)  # tanh 미분, dt = dh_next * (1 - h_next^2)
    db = np.sum(dt, axis=0)           # db = dt
    dWh = np.dot(h_prev.T, dt)        # dWh = h_prev * dt
    dh_prev = np.dot(dt, Wh.T)        # dh_prev = dt * Wh
    dWx = np.dot(x.T, dt)             # dWx = x * dt
    dx = np.dot(dt, Wx.T)             # dx = dt * Wx
    
    self.grads[0][...] = dWx          # dWx, dWh, db를 grads에 저장
    self.grads[1][...] = dWh
    self.grads[2][...] = db
    
    return dx, dh_prev                     # 입력 x와 이전 상태 h에 대한 기울기 dx, dh_prev 반환

### Time RNN 계층 구현    

In [ ]:
class TimeRNN:
  def __init__(self, Wx, Wh, b, stateful=False):   # Wx: 입력 x에 대한 가중치, Wh: 이전 상태 h에 대한 가중치, b: 편향, stateful: 상태 유지 여부
    self.params = [Wx, Wh, b]                      # Wx, Wh, b를 params에 저장
    self.grads = [np.zeros_like(Wx), np.zeros_like(Wh), np.zeros_like(b)]  # 기울기 초기화
    self.layers = None                             # 다수의 RNN 계층을 리스트로 저장
    
    self.h, self.dh = None, None                   # h는 forward() 메서드를 불렀을 때 마지막 RNN 계층의 은닉 상태를 저장, dh는 backward() 메서드를 불렀을 때 하나 앞 블록의 은닉 상태의 기울기를 저장
    self.stateful = stateful                       # 상태 유지 여부, 만약 False라면 은닉 상태를 영행렬(모든 요소가 0인 행렬)로 초기화
    
  def set_state(self, h):                          # 은닉 상태 설정
    self.h = h                                     # h에 입력된 h를 저장
    
  def reset_state(self):                           # 은닉 상태 초기화
    self.h = None                                  # h를 None으로 초기화
  
  # 순전파
  def forward(self, xs):                           # xs: T개 분량의 시계열 데이터를 하나로 모은 것
    Wx, Wh, b = self.params                        # Wx, Wh, b를 params에서 가져옴
    N, T, D = xs.shape                             # N: 미니배치 크기, T: 시계열 데이터의 길이, D: 입력 벡터의 차원 수
    D, H = Wx.shape                                # Wx의 형상을 가져옴
    
    self.layers = []                               # RNN 계층을 저장할 리스트 layers 초기화
    hs = np.empty((N, T, H), dtype='f')            # 은닉 상태를 저장할 hs를 영행렬로 초기화
    
    if not self.stateful or self.h is None:        # 상태 유지가 False이거나 은닉 상태가 None이면
      self.h = np.zeros((N, H), dtype='f')         # 은닉 상태를 영행렬로 초기화
    
    for t in range(T):                             # T번 반복
      layer = RNN(*self.params)                    # RNN 계층 생성
      self.h = layer.forward(xs[:, t, :], self.h)  # RNN 계층의 forward 메서드 호출, 은닉 상태 구한 후 h에 저장
      hs[:, t, :] = self.h                         # hs에 계산한 은닉 상태 저장
      self.layers.append(layer)                    # RNN 계층을 layers에 추가
    
    return hs                                     # hs 반환 -> 행렬 hs에는 각 시각의 은닉 상태(h)가 저장되어 있음, 총 T개의 은닉 상태(h)가 저장되어 있음
  
  # 역전파
  def backward(self, dhs):
    Wx, Wh, b = self.params                        # Wx, Wh, b를 params에서 가져옴
    N, T, H = dhs.shape                            # dhs의 형상을 가져옴
    D, H = Wx.shape                                # Wx의 형상을 가져옴
    
    dxs = np.empty((N, T, D), dtype='f')           # dxs를 영행렬로 초기화
    dh = 0                                         # dh를 0으로 초기화
    grads = [0, 0, 0]                              # grads를 0으로 초기화
    
    for t in reversed(range(T)):                   # T-1부터 0까지 반복
      layer = self.layers[t]                       # t번째 RNN 계층을 가져옴
      dx, dh = layer.backward(dhs[:, t, :] + dh)   # RNN 계층의 backward 메서드 호출, dx와 dh를 구함
      dxs[:, t, :] = dx                            # dxs에 dx 저장
      
      for i, grad in enumerate(layer.grads):       # layer.grads에는 dWx, dWh, db가 저장되어 있음
        grads[i] += grad                          # grads에 grad를 더함
    
    for i, grad in enumerate(grads):               # grads에 저장된 기울기를 각각 꺼내어
      self.grads[i][...] = grad                    # self.grads에 저장, 각 RNN 계층의 가중치 기울기를 모두 더한 것이 grads이므로 이를 self.grads에 저장
    
    self.dh = dh                                   # dh를 self.dh에 저장
    
    return dxs                                     # dxs 반환